# Cerebellar probability network demo

This notebook builds one configurable cerebellar network with probabilistic projections. Each projection places one named `ExpSyn` pool at the postsynaptic soma midpoint, runs the network once, reports timing/edge summaries, and plots one spike raster per cell type.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5" 

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import sys
import time
from dataclasses import dataclass
from pathlib import Path


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import brainstate
import brainunit as u
import jax
import matplotlib.pyplot as plt
import numpy as np

import braincell
from braincell import mech
from braincell.filter import at

from examples.neuron_compare.cell.bc_ma2025.bc_braincell import BC as BrainCellBC
from examples.neuron_compare.cell.bc_ma2025.parameters import DEFAULT_MORPH_PATH as BC_MORPH_PATH
from examples.neuron_compare.cell.bc_ma2025.parameters import load_bc25_params
from examples.neuron_compare.cell.dcn_su2015.dcn_braincell import DCN as BrainCellDCN
from examples.neuron_compare.cell.dcn_su2015.parameters import load_dcn15_params
from examples.neuron_compare.cell.goc_ma2020.goc_braincell import GoC as BrainCellGoC
from examples.neuron_compare.cell.goc_ma2020.parameters import DEFAULT_MORPH_PATH as GOC_MORPH_PATH
from examples.neuron_compare.cell.goc_ma2020.parameters import load_goc20_params
from examples.neuron_compare.cell.grc_ma2020.grc_braincell import GrC as BrainCellGrC
from examples.neuron_compare.cell.grc_ma2020.parameters import DEFAULT_MORPH_PATH as GRC_MORPH_PATH
from examples.neuron_compare.cell.grc_ma2020.parameters import load_grc20_params
from examples.neuron_compare.cell.io_zh2019.io_braincell import IO as BrainCellIO
from examples.neuron_compare.cell.io_zh2019.parameters import load_io19_params
from examples.neuron_compare.cell.pc_ma2024.parameters import DEFAULT_MORPH_PATH as PC_MORPH_PATH
from examples.neuron_compare.cell.pc_ma2024.parameters import load_pc24_params
from examples.neuron_compare.cell.pc_ma2024.pc_braincell import PC as BrainCellPC
from examples.neuron_compare.cell.sc_ma2021.parameters import DEFAULT_MORPH_PATH as SC_MORPH_PATH
from examples.neuron_compare.cell.sc_ma2021.parameters import load_sc21_params
from examples.neuron_compare.cell.sc_ma2021.sc_braincell import SC as BrainCellSC

brainstate.environ.set(precision=64)

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)
print("JAX backend:", jax.default_backend())
print("JAX devices:", [str(device) for device in jax.devices()])

In [ ]:
@dataclass(frozen=True)
class CellSpec:
    name: str
    cls: type
    params_loader: object
    temperature_celsius: float
    morph_path: object = None


CELL_SPECS = {
    "GrC": CellSpec("GrC", BrainCellGrC, load_grc20_params, 25.0, GRC_MORPH_PATH),
    "GoC": CellSpec("GoC", BrainCellGoC, load_goc20_params, 34.0, GOC_MORPH_PATH),
    "PC": CellSpec("PC", BrainCellPC, load_pc24_params, 36.0, PC_MORPH_PATH),
    "SC": CellSpec("SC", BrainCellSC, load_sc21_params, 32.0, SC_MORPH_PATH),
    "BC": CellSpec("BC", BrainCellBC, load_bc25_params, 36.0, BC_MORPH_PATH),
    "DCN": CellSpec("DCN", BrainCellDCN, lambda: load_dcn15_params(temperature_celsius=32.0), 32.0),
    "IO": CellSpec("IO", BrainCellIO, load_io19_params, 36.0),
}

# Edit these sizes first. Keep the first run small, then scale up.
CELL_SIZES = {
    "GrC": 16,
    "GoC": 16,
    "PC": 36,
    "SC": 24,
    "BC": 24,
    "DCN": 8,
    "IO": 24,
}
DT = 0.1 * u.ms
DURATION = 100* u.ms
V_INIT_MV = -65.0
V_THRESHOLD = 0.0 * u.mV
EVENT_BACKEND = "auto"
BRAINEVENT_BACKEND = "jax_raw"
RAISE_ON_ERROR = True

DEFAULT_P = 0.1
DEFAULT_WEIGHT = 0.005 * u.uS
DEFAULT_DELAY = 0.5 * u.ms
DEFAULT_TAU = 2.0 * u.ms

PROJECTION_CONFIGS = [
    {"pre": "GrC", "post": "GoC", "p": 0.2, "seed": 101, "weight": 0.002* u.uS, "delay": DEFAULT_DELAY, "e": 0.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "GoC", "post": "GrC", "p": DEFAULT_P, "seed": 102, "weight": 0.00014* u.uS, "delay": DEFAULT_DELAY, "e": -75.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "GrC", "post": "PC", "p": DEFAULT_P, "seed": 103, "weight": 0.006* u.uS, "delay": DEFAULT_DELAY, "e": 0.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "GrC", "post": "SC", "p": 0.2, "seed": 104, "weight": 0.00045* u.uS, "delay": DEFAULT_DELAY, "e": 0.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "GrC", "post": "BC", "p": 0.2, "seed": 105, "weight": 0.00035* u.uS, "delay": DEFAULT_DELAY, "e": 0.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "SC", "post": "PC", "p": DEFAULT_P, "seed": 106, "weight": 0.025* u.uS, "delay": DEFAULT_DELAY, "e": -75.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "BC", "post": "PC", "p": DEFAULT_P, "seed": 107, "weight": 0.02* u.uS, "delay": DEFAULT_DELAY, "e": -75.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "PC", "post": "DCN", "p": DEFAULT_P, "seed": 108, "weight": 0.006* u.uS, "delay": DEFAULT_DELAY, "e": -75.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "IO", "post": "PC", "p": 0.05, "seed": 109, "weight": 0.005*5* u.uS, "delay": DEFAULT_DELAY, "e": 0.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "DCN", "post": "IO", "p": 0.05, "seed": 110, "weight": 0.0003* u.uS, "delay": DEFAULT_DELAY, "e": -75.0 * u.mV, "tau": DEFAULT_TAU},
    {"pre": "IO", "post": "DCN", "p": 0.05, "seed": 111, "weight": 0.0025*2* u.uS, "delay": DEFAULT_DELAY, "e": 0.0 * u.mV, "tau": DEFAULT_TAU},
]

# Optional external drive. Set amplitude to 0 nA to disable a drive.
DRIVE_CONFIGS = {
    "GrC": {"delay": 0.1 * u.ms, "duration": 50 * u.ms, "amplitude": 0.1 * u.nA},
    "IO": {"delay": 0.1 * u.ms, "duration": 50 * u.ms, "amplitude": 0.1 * u.nA},
}

print("CELL_SIZES:", CELL_SIZES)
print("dt:", DT, "duration:", DURATION)
print("projection count:", len(PROJECTION_CONFIGS))

In [ ]:
def synapse_name(pre: str, post: str) -> str:
    return f"syn_{pre}_to_{post}"


def load_params(spec: CellSpec):
    return spec.params_loader()


def drive_protocol(pop_name: str, size: int):
    cfg = DRIVE_CONFIGS.get(pop_name)
    if cfg is None:
        return None
    amp_nA = float(np.asarray(cfg["amplitude"].to_decimal(u.nA)))
    if amp_nA == 0.0:
        return None
    amplitudes = u.Quantity(np.full(size, amp_nA, dtype=float), u.nA)
    delays = u.Quantity(np.full(size, float(np.asarray(cfg["delay"].to_decimal(u.ms))), dtype=float), u.ms)
    return mech.CurrentClamp(delay=delays, durations=cfg["duration"], amplitudes=amplitudes)


def incoming_projection_configs(pop_name: str):
    return [cfg for cfg in PROJECTION_CONFIGS if cfg["post"] == pop_name]


def build_population(spec: CellSpec, size: int, incoming_configs):
    params = load_params(spec)
    kwargs = {
        "params": params,
        "temperature_celsius": spec.temperature_celsius,
        "v_init_mV": V_INIT_MV,
        "pop_size": (size,),
        "name": f"{spec.name.lower()}_probability_network_{size}",
    }
    if spec.morph_path is None:
        assembly = spec.cls(**kwargs).build()
    else:
        assembly = spec.cls(spec.morph_path, **kwargs).build()

    cell = assembly.cell
    cell.V_th = V_THRESHOLD
    cell.place(at("soma", 0.5), mech.StateProbe(name="v", field="v"))

    drive = drive_protocol(spec.name, size)
    if drive is not None:
        cell.place(at("soma", 0.5), drive)

    for cfg in incoming_configs:
        name = synapse_name(cfg["pre"], cfg["post"])
        cell.place(
            at("soma", 0.5),
            mech.Synapse(
                "ExpSyn",
                tau=cfg["tau"],
                e=cfg["e"],
                weight=1.0 * u.uS,
                name=name,
            ),
        )
    return cell


def block_until_ready_tree(value):
    if hasattr(value, "block_until_ready"):
        value.block_until_ready()
        return
    if isinstance(value, dict):
        for item in value.values():
            block_until_ready_tree(item)
        return
    if isinstance(value, (tuple, list)):
        for item in value:
            block_until_ready_tree(item)
        return
    if hasattr(value, "mantissa"):
        block_until_ready_tree(value.mantissa)


def sync_network_result(result):
    block_until_ready_tree(result.time)
    block_until_ready_tree(result.traces)
    block_until_ready_tree(result.spikes)


def spikes_2d(value):
    arr = np.asarray(value, dtype=float)
    return arr.reshape((arr.shape[0], -1))

In [ ]:
total_start = time.perf_counter()

build_rows = []
populations = {}

for pop_name, spec in CELL_SPECS.items():
    size = int(CELL_SIZES[pop_name])
    if size <= 0:
        raise ValueError(f"CELL_SIZES[{pop_name!r}] must be positive, got {size}.")
    start = time.perf_counter()
    populations[pop_name] = build_population(spec, size, incoming_projection_configs(pop_name))
    build_rows.append({"cell": pop_name, "N": size, "build_cell_s": time.perf_counter() - start})
    print("built", pop_name, "N=", size)

init_start = time.perf_counter()
for cell in populations.values():
    cell.init_state()
    cell.reset_state()
init_reset_seconds = time.perf_counter() - init_start

network_start = time.perf_counter()
net = braincell.Network(name="cerebellar_probability_network")
for pop_name, cell in populations.items():
    net.add_population(pop_name, cell)

projection_rows = []
for cfg in PROJECTION_CONFIGS:
    pre = cfg["pre"]
    post = cfg["post"]
    edge_name = f"edges_{pre}_to_{post}"
    projection_name = f"proj_{pre}_to_{post}"
    edges = net.add_edges(
        name=edge_name,
        pre=pre,
        post=post,
        method=braincell.network.probability(p=float(cfg["p"]), seed=int(cfg["seed"]), allow_self=True),
    )
    if edges.n_edge > 0:
        weights = np.full(edges.n_edge, float(np.asarray(cfg["weight"].to_decimal(u.uS))), dtype=float) * u.uS
        net.add_projection(
            name=projection_name,
            edges=edge_name,
            synapse=synapse_name(pre, post),
            weight=weights,
            delay=cfg["delay"],
        )
    projection_rows.append(
        {
            "projection": f"{pre}->{post}",
            "p": float(cfg["p"]),
            "seed": int(cfg["seed"]),
            "edges": int(edges.n_edge),
            "e_mV": float(np.asarray(cfg["e"].to_decimal(u.mV))),
            "weight_uS": float(np.asarray(cfg["weight"].to_decimal(u.uS))),
            "delay_ms": float(np.asarray(cfg["delay"].to_decimal(u.ms))),
        }
    )
network_build_seconds = time.perf_counter() - network_start

run_start = time.perf_counter()
result = net.run(
    dt=DT,
    duration=DURATION,
    event_backend=EVENT_BACKEND,
    brainevent_backend=BRAINEVENT_BACKEND,
)
sync_network_result(result)
run_seconds = time.perf_counter() - run_start
total_seconds = time.perf_counter() - total_start

steps = int(result.time.shape[0])
spike_rows = []
for pop_name in CELL_SPECS:
    spikes = spikes_2d(result.spikes[pop_name])
    spike_rows.append({"cell": pop_name, "N": int(CELL_SIZES[pop_name]), "spike_count": int(spikes.sum())})

timing_summary = {
    "steps": steps,
    "init_reset_s": init_reset_seconds,
    "network_build_s": network_build_seconds,
    "run_s": run_seconds,
    "run_ms_per_step": 1000.0 * run_seconds / max(steps, 1),
    "total_s": total_seconds,
}

print("timing:", timing_summary)

In [ ]:
try:
    import pandas as pd

    print("cell build times")
    display(pd.DataFrame(build_rows))
    print("projection edges")
    display(pd.DataFrame(projection_rows))
    print("spike counts")
    display(pd.DataFrame(spike_rows))
    print("timing")
    display(pd.DataFrame([timing_summary]))
except Exception:
    print("cell build times", build_rows)
    print("projection edges", projection_rows)
    print("spike counts", spike_rows)
    print("timing", timing_summary)

In [ ]:
time_ms = np.asarray(result.time.to_decimal(u.ms), dtype=float)
cell_names = tuple(CELL_SPECS)
ncols = 2
nrows = int(np.ceil(len(cell_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 2.8 * nrows), squeeze=False, sharex=True)

for axis in axes.ravel():
    axis.set_visible(False)

for axis, pop_name in zip(axes.ravel(), cell_names):
    axis.set_visible(True)
    voltage = np.asarray(result.traces[pop_name]["v"].to_decimal(u.mV), dtype=float)
    voltage = voltage.reshape((voltage.shape[0], -1))
    n_plot = min(5, voltage.shape[1])
    for cell_index in range(n_plot):
        axis.plot(time_ms, voltage[:, cell_index], linewidth=1.2, label=f"cell {cell_index}")
    axis.set_title(f"{pop_name} soma voltage")
    axis.set_ylabel("V soma (mV)")
    axis.grid(True, alpha=0.25)
    if n_plot > 1:
        axis.legend(loc="best", fontsize=8, frameon=False)
    if n_plot == 0:
        axis.text(0.5, 0.5, "no voltage traces", transform=axis.transAxes, ha="center", va="center", color="0.45")

for axis in axes[-1, :]:
    axis.set_xlabel("time (ms)")

fig.suptitle("Cerebellar probability network soma voltage traces")
fig.tight_layout()
plt.show()
